# 04 — Data Engineer B.I: Generación de Transacciones y Carga

Este notebook es **independiente** y no modifica los notebooks de los compañeros.

Compatibilidad verificada con:

- `01_setup_bigquery.ipynb`: usa exactamente las tablas y columnas creadas allí.
- `02_generate_data.ipynb`: reutiliza los CSV maestros generados allí (`categories.csv`, `products.csv`, `customers.csv`).

Qué hace este notebook:

1. Lee los datos maestros ya generados por el equipo.
2. Genera las tablas transaccionales:
   - `orders`
   - `order_items`
   - `payments`
   - `reviews`
3. Valida integridad referencial antes de cargar.
4. Exporta los CSV transaccionales a `data/`.
5. Carga las 7 tablas en BigQuery respetando el schema del notebook `01_setup_bigquery.ipynb`.

Orden recomendado de ejecución:

1. `02_generate_data.ipynb`
2. `01_setup_bigquery.ipynb`
3. `04_data_engineer_b_transactions_load.ipynb`


## 1. Imports y configuración

La ruta de datos se detecta automáticamente para funcionar tanto desde `notebooks/` como desde la raíz del repo.

In [ ]:
import os
import uuid
import random
from pathlib import Path
from datetime import timedelta

import pandas as pd
from faker import Faker
from dotenv import load_dotenv

# BigQuery
from google.cloud import bigquery
from google.oauth2 import service_account

fake = Faker("es_ES")
random.seed(42)
Faker.seed(42)

# Detecta carpeta data/ si el notebook se ejecuta desde notebooks/ o desde raíz
cwd = Path.cwd()
if (cwd / "data").exists() or cwd.name != "notebooks":
    DATA_DIR = cwd / "data"
else:
    DATA_DIR = cwd.parent / "data"

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Carpeta de datos: {DATA_DIR.resolve()}")


## 2. Leer datos maestros generados por Ana

Este notebook **no regenera** `customers`, `products` ni `categories`; los reutiliza para no pisar el trabajo de los compañeros.

In [ ]:
required_master_files = {
    "categories": DATA_DIR / "categories.csv",
    "products": DATA_DIR / "products.csv",
    "customers": DATA_DIR / "customers.csv",
}

missing = [str(path) for path in required_master_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Faltan CSVs maestros. Ejecuta primero 02_generate_data.ipynb.\n"
        + "\n".join(missing)
    )

df_categories = pd.read_csv(required_master_files["categories"], parse_dates=["created_at"])
df_products = pd.read_csv(required_master_files["products"], parse_dates=["created_at"])
df_customers = pd.read_csv(required_master_files["customers"], parse_dates=["registered_at"])

print("Datos maestros cargados:")
print(f"categories: {len(df_categories)} filas")
print(f"products  : {len(df_products)} filas")
print(f"customers : {len(df_customers)} filas")


## 2.1. Comprobación de compatibilidad con `01` y `02`

Antes de generar transacciones, se comprueba que los CSV maestros tienen las columnas exactas que necesita este notebook y que coinciden con las tablas definidas en `01_setup_bigquery.ipynb`.


In [ ]:
EXPECTED_MASTER_COLUMNS = {
    "categories": ["category_id", "name", "description", "created_at"],
    "products": ["product_id", "category_id", "name", "description", "brand", "price", "cost", "stock", "is_active", "created_at"],
    "customers": ["customer_id", "first_name", "last_name", "email", "phone", "city", "country", "acquisition_channel", "registered_at", "is_active"],
}

master_dfs = {
    "categories": df_categories,
    "products": df_products,
    "customers": df_customers,
}

for table_name, expected_cols in EXPECTED_MASTER_COLUMNS.items():
    actual_cols = list(master_dfs[table_name].columns)
    missing_cols = [col for col in expected_cols if col not in actual_cols]
    extra_cols = [col for col in actual_cols if col not in expected_cols]
    if missing_cols:
        raise ValueError(f"{table_name}.csv no es compatible. Faltan columnas: {missing_cols}")
    if extra_cols:
        print(f"Aviso: {table_name}.csv tiene columnas extra que no se cargarán en BigQuery: {extra_cols}")

print("CSV maestros compatibles con el schema esperado.")


## 3. Enumerados de transacciones

Los estados y métodos están controlados para evitar valores inconsistentes en BigQuery.

In [ ]:
ORDER_STATUSES = ["pending", "processing", "shipped", "delivered", "cancelled"]
ORDER_STATUS_WEIGHTS = [0.08, 0.10, 0.14, 0.58, 0.10]

PAYMENT_METHODS = ["card", "paypal", "bank_transfer", "bizum"]
PAYMENT_STATUS_BY_ORDER_STATUS = {
    "pending": ["pending", "failed"],
    "processing": ["paid"],
    "shipped": ["paid"],
    "delivered": ["paid"],
    "cancelled": ["refunded", "failed"],
}

REVIEW_COMMENTS = {
    1: ["Muy mala experiencia", "No lo recomendaría", "El producto no cumplió expectativas"],
    2: ["Regular", "Podría mejorar", "Calidad inferior a la esperada"],
    3: ["Correcto", "Cumple su función", "Producto aceptable"],
    4: ["Buen producto", "Buena relación calidad-precio", "Recomendable"],
    5: ["Excelente", "Muy satisfecho", "Lo volvería a comprar"],
}


## 4. Generar `orders`

Cada pedido usa un `customer_id` existente. Las fechas son coherentes: `ordered_at <= shipped_at <= delivered_at` cuando aplica.

In [ ]:
def generar_orders(df_customers: pd.DataFrame, n_orders: int = 2000) -> pd.DataFrame:
    orders = []
    customer_ids = df_customers["customer_id"].tolist()
    customer_lookup = df_customers.set_index("customer_id")

    for _ in range(n_orders):
        customer_id = random.choice(customer_ids)
        customer = customer_lookup.loc[customer_id]
        status = random.choices(ORDER_STATUSES, weights=ORDER_STATUS_WEIGHTS, k=1)[0]

        # Fecha posterior a registro del cliente
        registered_at = pd.to_datetime(customer["registered_at"])
        min_order_date = max(registered_at, pd.Timestamp.now() - pd.Timedelta(days=730))
        ordered_at = fake.date_time_between(start_date=min_order_date.to_pydatetime(), end_date="now")
        ordered_at = pd.Timestamp(ordered_at)

        shipped_at = pd.NaT
        delivered_at = pd.NaT

        if status in ["shipped", "delivered"]:
            shipped_at = ordered_at + pd.Timedelta(days=random.randint(1, 5), hours=random.randint(0, 23))

        if status == "delivered":
            delivered_at = shipped_at + pd.Timedelta(days=random.randint(1, 7), hours=random.randint(0, 23))

        orders.append({
            "order_id": str(uuid.uuid4()),
            "customer_id": customer_id,
            "status": status,
            "shipping_address": fake.address().replace("\n", ", "),
            "shipping_city": customer["city"],
            "shipping_country": customer["country"],
            "ordered_at": ordered_at,
            "shipped_at": shipped_at,
            "delivered_at": delivered_at,
        })

    return pd.DataFrame(orders)


df_orders = generar_orders(df_customers, n_orders=2000)
df_orders.head()


## 5. Generar `order_items`

Cada pedido tiene entre 1 y 4 líneas. Cada línea usa un `product_id` existente y mantiene el precio histórico en `unit_price`.

In [ ]:
def generar_order_items(df_orders: pd.DataFrame, df_products: pd.DataFrame) -> pd.DataFrame:
    order_items = []
    products = df_products[["product_id", "price"]].to_dict("records")

    for order_id in df_orders["order_id"]:
        n_items = random.choices([1, 2, 3, 4], weights=[0.50, 0.30, 0.15, 0.05], k=1)[0]
        selected_products = random.sample(products, k=n_items)

        for product in selected_products:
            quantity = random.choices([1, 2, 3, 4, 5], weights=[0.55, 0.25, 0.12, 0.05, 0.03], k=1)[0]
            base_price = float(product["price"])
            # Pequeña variación histórica sobre el PVP actual
            unit_price = round(base_price * random.uniform(0.90, 1.05), 2)
            discount = random.choices([0, 0.05, 0.10, 0.15, 0.20], weights=[0.65, 0.14, 0.10, 0.07, 0.04], k=1)[0]

            order_items.append({
                "order_item_id": str(uuid.uuid4()),
                "order_id": order_id,
                "product_id": product["product_id"],
                "quantity": quantity,
                "unit_price": unit_price,
                "discount": discount,
            })

    return pd.DataFrame(order_items)


df_order_items = generar_order_items(df_orders, df_products)
df_order_items.head()


## 6. Generar `payments`

El importe se calcula desde `order_items`: `quantity * unit_price * (1 - discount)`.

In [ ]:
def calcular_importes_pedido(df_order_items: pd.DataFrame) -> pd.DataFrame:
    tmp = df_order_items.copy()
    tmp["line_total"] = tmp["quantity"] * tmp["unit_price"] * (1 - tmp["discount"])
    totals = tmp.groupby("order_id", as_index=False)["line_total"].sum()
    totals["amount"] = totals["line_total"].round(2)
    return totals[["order_id", "amount"]]


def generar_payments(df_orders: pd.DataFrame, df_order_items: pd.DataFrame) -> pd.DataFrame:
    order_amounts = calcular_importes_pedido(df_order_items)
    orders_with_amounts = df_orders.merge(order_amounts, on="order_id", how="left")

    payments = []
    for _, order in orders_with_amounts.iterrows():
        possible_statuses = PAYMENT_STATUS_BY_ORDER_STATUS[order["status"]]
        if order["status"] == "pending":
            payment_status = random.choices(possible_statuses, weights=[0.85, 0.15], k=1)[0]
        elif order["status"] == "cancelled":
            payment_status = random.choices(possible_statuses, weights=[0.75, 0.25], k=1)[0]
        else:
            payment_status = "paid"

        paid_at = pd.NaT
        if payment_status in ["paid", "refunded"]:
            paid_at = pd.to_datetime(order["ordered_at"]) + pd.Timedelta(minutes=random.randint(1, 180))

        payments.append({
            "payment_id": str(uuid.uuid4()),
            "order_id": order["order_id"],
            "method": random.choice(PAYMENT_METHODS),
            "status": payment_status,
            "amount": float(order["amount"]),
            "paid_at": paid_at,
        })

    return pd.DataFrame(payments)


df_payments = generar_payments(df_orders, df_order_items)
df_payments.head()


## 7. Generar `reviews`

Solo se crean reviews para pedidos entregados, porque son compras que el cliente ya ha recibido.

In [ ]:
def generar_reviews(df_orders: pd.DataFrame, df_order_items: pd.DataFrame, review_rate: float = 0.35) -> pd.DataFrame:
    delivered_orders = set(df_orders.loc[df_orders["status"] == "delivered", "order_id"])
    delivered_items = df_order_items[df_order_items["order_id"].isin(delivered_orders)].copy()

    reviews = []
    for _, item in delivered_items.iterrows():
        if random.random() <= review_rate:
            rating = random.choices([1, 2, 3, 4, 5], weights=[0.04, 0.08, 0.18, 0.35, 0.35], k=1)[0]
            order = df_orders.loc[df_orders["order_id"] == item["order_id"]].iloc[0]
            delivered_at = pd.to_datetime(order["delivered_at"])
            created_at = delivered_at + pd.Timedelta(days=random.randint(1, 45), hours=random.randint(0, 23))

            reviews.append({
                "review_id": str(uuid.uuid4()),
                "order_item_id": item["order_item_id"],
                "rating": rating,
                "comment": random.choice(REVIEW_COMMENTS[rating]),
                "created_at": created_at,
            })

    return pd.DataFrame(reviews)


df_reviews = generar_reviews(df_orders, df_order_items, review_rate=0.35)
df_reviews.head()


## 8. Validaciones de calidad e integridad

Estas comprobaciones detectan errores antes de exportar o cargar a BigQuery.

In [ ]:
print("=" * 60)
print("RESUMEN DE TABLAS")
print("=" * 60)
for name, df in {
    "categories": df_categories,
    "customers": df_customers,
    "products": df_products,
    "orders": df_orders,
    "order_items": df_order_items,
    "payments": df_payments,
    "reviews": df_reviews,
}.items():
    print(f"{name:<12}: {len(df):>6} filas")

# Primary keys únicas
assert df_orders["order_id"].is_unique, "order_id duplicado"
assert df_order_items["order_item_id"].is_unique, "order_item_id duplicado"
assert df_payments["payment_id"].is_unique, "payment_id duplicado"
assert df_reviews["review_id"].is_unique, "review_id duplicado"

# Foreign keys válidas
assert df_orders["customer_id"].isin(df_customers["customer_id"]).all(), "orders.customer_id inválido"
assert df_order_items["order_id"].isin(df_orders["order_id"]).all(), "order_items.order_id inválido"
assert df_order_items["product_id"].isin(df_products["product_id"]).all(), "order_items.product_id inválido"
assert df_payments["order_id"].isin(df_orders["order_id"]).all(), "payments.order_id inválido"
assert df_reviews["order_item_id"].isin(df_order_items["order_item_id"]).all(), "reviews.order_item_id inválido"

# Coherencia negocio
assert (df_order_items["quantity"] > 0).all(), "quantity debe ser > 0"
assert (df_order_items["unit_price"] > 0).all(), "unit_price debe ser > 0"
assert df_order_items["discount"].between(0, 1).all(), "discount fuera de rango"
assert (df_payments["amount"] > 0).all(), "amount debe ser > 0"
assert df_reviews["rating"].between(1, 5).all(), "rating fuera de rango"

# Fechas coherentes
mask_shipped = df_orders["shipped_at"].notna()
mask_delivered = df_orders["delivered_at"].notna()
assert (pd.to_datetime(df_orders.loc[mask_shipped, "shipped_at"]) >= pd.to_datetime(df_orders.loc[mask_shipped, "ordered_at"])).all()
assert (pd.to_datetime(df_orders.loc[mask_delivered, "delivered_at"]) >= pd.to_datetime(df_orders.loc[mask_delivered, "shipped_at"])).all()

print("\nValidaciones OK")


## 9. Exportar CSVs transaccionales

Se guardan en la misma carpeta `data/` donde están los CSVs maestros.

In [ ]:
transactional_tables = {
    "orders": df_orders,
    "order_items": df_order_items,
    "payments": df_payments,
    "reviews": df_reviews,
}

for table_name, df in transactional_tables.items():
    path = DATA_DIR / f"{table_name}.csv"
    df.to_csv(path, index=False)
    print(f"{path.name:<20} -> {len(df):>6} filas exportadas en {path}")


## 10. Conexión a BigQuery

Usa el mismo `.env` del proyecto:

```text
GCP_PROJECT_ID=...
BQ_DATASET_ID=...
GOOGLE_APPLICATION_CREDENTIALS=...
```

Si el `.env` está en la raíz del repo, también se detecta automáticamente al ejecutar desde `notebooks/`.

In [ ]:
# Detecta .env desde notebooks/ o desde raíz
possible_env_paths = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
for env_path in possible_env_paths:
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env cargado desde: {env_path}")
        break
else:
    load_dotenv()
    print("No se encontró .env explícito; se intentan usar variables de entorno del sistema")

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

if not PROJECT_ID or not DATASET_ID:
    raise ValueError("Faltan GCP_PROJECT_ID o BQ_DATASET_ID en .env")

# Resolver ruta de credenciales si es relativa
if CREDENTIALS_PATH:
    cred_path = Path(CREDENTIALS_PATH)
    if not cred_path.is_absolute():
        candidates = [Path.cwd() / cred_path, Path.cwd().parent / cred_path]
        for candidate in candidates:
            if candidate.exists():
                CREDENTIALS_PATH = str(candidate.resolve())
                break

if CREDENTIALS_PATH and Path(CREDENTIALS_PATH).exists():
    credentials = service_account.Credentials.from_service_account_file(CREDENTIALS_PATH)
    client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
    print("Cliente BigQuery creado con Service Account")
else:
    client = bigquery.Client(project=PROJECT_ID)
    print("Cliente BigQuery creado con credenciales por defecto")

print(f"Proyecto: {PROJECT_ID}")
print(f"Dataset : {DATASET_ID}")


## 11. Carga a BigQuery

Carga las 7 tablas en orden de dependencias. Usa `WRITE_TRUNCATE` para que la ejecución sea reproducible.

In [ ]:
# Schema exacto alineado con 01_setup_bigquery.ipynb
BQ_SCHEMAS = {
    "categories": [
        bigquery.SchemaField("category_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("description", "STRING"),
        bigquery.SchemaField("created_at", "TIMESTAMP"),
    ],
    "customers": [
        bigquery.SchemaField("customer_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("first_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("last_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("email", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("phone", "STRING"),
        bigquery.SchemaField("city", "STRING"),
        bigquery.SchemaField("country", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("acquisition_channel", "STRING"),
        bigquery.SchemaField("registered_at", "TIMESTAMP", mode="REQUIRED"),
        bigquery.SchemaField("is_active", "BOOLEAN"),
    ],
    "products": [
        bigquery.SchemaField("product_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("category_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("description", "STRING"),
        bigquery.SchemaField("brand", "STRING"),
        bigquery.SchemaField("price", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("cost", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("stock", "INTEGER"),
        bigquery.SchemaField("is_active", "BOOLEAN"),
        bigquery.SchemaField("created_at", "TIMESTAMP"),
    ],
    "orders": [
        bigquery.SchemaField("order_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("customer_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("status", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("shipping_address", "STRING"),
        bigquery.SchemaField("shipping_city", "STRING"),
        bigquery.SchemaField("shipping_country", "STRING"),
        bigquery.SchemaField("ordered_at", "TIMESTAMP", mode="REQUIRED"),
        bigquery.SchemaField("shipped_at", "TIMESTAMP"),
        bigquery.SchemaField("delivered_at", "TIMESTAMP"),
    ],
    "order_items": [
        bigquery.SchemaField("order_item_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("order_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("product_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("quantity", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("unit_price", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("discount", "FLOAT"),
    ],
    "payments": [
        bigquery.SchemaField("payment_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("order_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("method", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("status", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("amount", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("paid_at", "TIMESTAMP"),
    ],
    "reviews": [
        bigquery.SchemaField("review_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("order_item_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("rating", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("comment", "STRING"),
        bigquery.SchemaField("created_at", "TIMESTAMP"),
    ],
}

TIMESTAMP_COLUMNS = {
    "categories": ["created_at"],
    "customers": ["registered_at"],
    "products": ["created_at"],
    "orders": ["ordered_at", "shipped_at", "delivered_at"],
    "payments": ["paid_at"],
    "reviews": ["created_at"],
}

INTEGER_COLUMNS = {
    "products": ["stock"],
    "order_items": ["quantity"],
    "reviews": ["rating"],
}

FLOAT_COLUMNS = {
    "products": ["price", "cost"],
    "order_items": ["unit_price", "discount"],
    "payments": ["amount"],
}

BOOLEAN_COLUMNS = {
    "customers": ["is_active"],
    "products": ["is_active"],
}


def prepare_for_bq(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    """Ordena columnas y normaliza tipos para cargar contra el schema de 01_setup_bigquery.ipynb."""
    schema = BQ_SCHEMAS[table_name]
    expected_cols = [field.name for field in schema]
    missing_cols = [col for col in expected_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{table_name}: faltan columnas requeridas para BigQuery: {missing_cols}")

    df_bq = df[expected_cols].copy()

    for col in TIMESTAMP_COLUMNS.get(table_name, []):
        df_bq[col] = pd.to_datetime(df_bq[col], errors="coerce")

    for col in INTEGER_COLUMNS.get(table_name, []):
        df_bq[col] = pd.to_numeric(df_bq[col], errors="coerce").astype("Int64")

    for col in FLOAT_COLUMNS.get(table_name, []):
        df_bq[col] = pd.to_numeric(df_bq[col], errors="coerce").astype(float)

    for col in BOOLEAN_COLUMNS.get(table_name, []):
        df_bq[col] = df_bq[col].astype(bool)

    return df_bq


def load_dataframe_to_bq(df: pd.DataFrame, table_name: str) -> None:
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    df_bq = prepare_for_bq(df, table_name)
    job_config = bigquery.LoadJobConfig(
        schema=BQ_SCHEMAS[table_name],
        write_disposition="WRITE_TRUNCATE",
    )
    job = client.load_table_from_dataframe(df_bq, table_id, job_config=job_config)
    job.result()
    table = client.get_table(table_id)
    print(f"{table_name:<12} -> {table.num_rows:>6} filas cargadas en {table_id}")

# Orden compatible con dependencias lógicas del modelo definido en 01_setup_bigquery.ipynb
load_order = [
    ("categories", df_categories),
    ("customers", df_customers),
    ("products", df_products),
    ("orders", df_orders),
    ("order_items", df_order_items),
    ("payments", df_payments),
    ("reviews", df_reviews),
]

for table_name, df in load_order:
    load_dataframe_to_bq(df, table_name)


## 12. Verificación post-carga

Comprueba el número de filas de cada tabla ya cargada en BigQuery.

In [ ]:
query = f"""
SELECT 'categories' AS table_name, COUNT(*) AS total_rows FROM `{PROJECT_ID}.{DATASET_ID}.categories`
UNION ALL
SELECT 'customers', COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.customers`
UNION ALL
SELECT 'products', COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.products`
UNION ALL
SELECT 'orders', COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.orders`
UNION ALL
SELECT 'order_items', COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.order_items`
UNION ALL
SELECT 'payments', COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.payments`
UNION ALL
SELECT 'reviews', COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.reviews`
ORDER BY table_name
"""

client.query(query).to_dataframe()


## 13. Queries técnicas rápidas

Estas queries sirven para demostrar que la carga es coherente y que las relaciones funcionan.

In [ ]:
# Orders sin customer válido: debería devolver 0
query_invalid_customers = f"""
SELECT COUNT(*) AS invalid_orders
FROM `{PROJECT_ID}.{DATASET_ID}.orders` o
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.customers` c
  ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL
"""
client.query(query_invalid_customers).to_dataframe()


In [ ]:
# Revenue por país
query_revenue_country = f"""
SELECT
  o.shipping_country,
  COUNT(DISTINCT o.order_id) AS total_orders,
  ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2) AS revenue
FROM `{PROJECT_ID}.{DATASET_ID}.orders` o
JOIN `{PROJECT_ID}.{DATASET_ID}.order_items` oi
  ON o.order_id = oi.order_id
WHERE o.status != 'cancelled'
GROUP BY o.shipping_country
ORDER BY revenue DESC
"""
client.query(query_revenue_country).to_dataframe()
